# 🇸🇦 Adaptive AI Tourism Planner — Production ML Pipeline

This is the **single Colab notebook** for the final ML/data pipeline.

It is intentionally separated into two responsibilities:

1. **Forecast model:** one-month-ahead **city-total** KAPSARC POS demand forecasting.
2. **Recommendation context:** observed KAPSARC city-total demand + national tourism-sector demand + DataSaudi regional seasonality.

### Important data-granularity correction

The KAPSARC source contains:
- city totals where `Sectors == Total`, and
- national sector totals where `City == Total`.

It does **not** provide an observed `City × Sector` transaction cross in this table.  
This notebook therefore never fabricates a city-sector transaction value.

### Output

Everything is saved automatically to Google Drive under:

`MyDrive/SaudiTourismPlanner_ML/production_runs/run_<UTC timestamp>/`

A second folder called `latest_backend_artifacts/` is also refreshed so the artifacts can be copied directly into the FastAPI backend's `ml_artifacts/` folder.

In [ ]:
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "scikit-learn>=1.7,<1.9" "joblib>=1.4,<2" "requests>=2.31,<3" "openpyxl>=3.1,<4" "matplotlib>=3.8,<4"

In [ ]:
from __future__ import annotations

import io
import json
import math
import os
import re
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("✓ Imports ready")

In [ ]:
# ---------------------------------------------------------------------
# Google Drive / run folders
# ---------------------------------------------------------------------
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/SaudiTourismPlanner_ML")
    RUNTIME = "google_colab"
except Exception:
    ROOT = Path.cwd() / "SaudiTourismPlanner_ML"
    RUNTIME = "local_jupyter"

RUN_DIR = ROOT / "production_runs" / f"run_{RUN_ID}"
RAW_DIR = RUN_DIR / "raw"
PROCESSED_DIR = RUN_DIR / "processed"
MODEL_DIR = RUN_DIR / "models"
METRICS_DIR = RUN_DIR / "metrics"
PLOTS_DIR = RUN_DIR / "plots"
REPORTS_DIR = RUN_DIR / "reports"
BACKEND_ARTIFACTS_DIR = RUN_DIR / "backend_artifacts"
LATEST_BACKEND_ARTIFACTS_DIR = ROOT / "latest_backend_artifacts"
CACHE_DIR = ROOT / "data_cache"

for d in [
    RAW_DIR, PROCESSED_DIR, MODEL_DIR, METRICS_DIR, PLOTS_DIR,
    REPORTS_DIR, BACKEND_ARTIFACTS_DIR, LATEST_BACKEND_ARTIFACTS_DIR,
    CACHE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("Runtime:", RUNTIME)
print("Run:", RUN_ID)
print("Output:", RUN_DIR)

In [ ]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
KAPSARC_DATASET_ID = "point-of-sale-transactions-by-sector-and-city"
KAPSARC_DATASET_PAGE = (
    "https://data.kapsarc.org/explore/dataset/"
    "point-of-sale-transactions-by-sector-and-city/"
)

KAPSARC_EXPORT_CANDIDATES = [
    (
        "https://data.kapsarc.org/api/explore/v2.1/catalog/datasets/"
        f"{KAPSARC_DATASET_ID}/exports/csv"
        "?lang=en&timezone=Asia%2FRiyadh&use_labels=true"
    ),
    (
        "https://datasource.kapsarc.org/api/explore/v2.1/catalog/datasets/"
        f"{KAPSARC_DATASET_ID}/exports/csv"
        "?lang=en&timezone=Asia%2FRiyadh&use_labels=true"
    ),
]

DATASAUDI_API_BASE = "https://api.datasaudi.sa/tesseract"
DATASAUDI_DATASET_PAGE = "https://datasaudi.sa/en/data-explorer/datasets"

# Optional local source files. If these exist, the notebook uses them.
# Otherwise it tries the official public sources automatically.
KAPSARC_LOCAL_PATH = CACHE_DIR / "point-of-sale-transactions-by-sector-and-city.csv"
DATASAUDI_LOCAL_XLSX = CACHE_DIR / "Occupancy_rates_all_Provinces_2024.xlsx"

VALUE_INDICATOR = "Value of Transactions (In Thousand SAR)"
TOURISM_SECTORS = {
    "Hotels": "accommodation",
    "Recreation and Culture": "attractions",
    "Restaurants & Café": "food",
}

CITY_NAME_MAP = {
    "ABHA": "Abha",
    "BURAIDAH": "Buraidah",
    "DAMMAM": "Dammam",
    "HAIL": "Hail",
    "JEDDAH": "Jeddah",
    "KHOBAR": "Khobar",
    "MADINA": "Madinah",
    "MAKKAH": "Makkah",
    "RIYADH": "Riyadh",
    "TABOUK": "Tabuk",
}

FORECAST_FEATURES = [
    "group_key",
    "year",
    "month_num",
    "month_sin",
    "month_cos",
    "trend_months",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",
    "rolling_3",
    "rolling_6",
    "rolling_12",
    "rolling_std_3",
    "rolling_std_6",
    "rolling_std_12",
]

FINAL_HOLDOUT_MONTHS = 12
VALIDATION_FOLDS = 3
VALIDATION_MONTHS_PER_FOLD = 6
MIN_WEEKLY_OBS_PER_MONTH = 4

RANK_WEIGHTS = {
    "preference_match": 0.45,
    "place_quality": 0.10,
    "regional_demand": 0.15,
    "seasonality": 0.10,
    "distance_fit": 0.20,
}

print("✓ Configuration ready")

In [ ]:
# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def save_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)

def normalize_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    s = str(value).strip().lower()
    s = re.sub(r"[\u200e\u200f]", "", s)
    s = re.sub(r"[^a-z0-9\u0600-\u06ff]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def minmax01(series: pd.Series, neutral: float = 0.5) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or np.isclose(lo, hi):
        return pd.Series(neutral, index=series.index, dtype=float)
    return ((s - lo) / (hi - lo)).clip(0, 1)

def safe_mape(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_true) > 1e-9
    if not mask.any():
        return float("nan")
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])))

def regression_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "r2": float(r2_score(y_true, y_pred)),
        "mape": safe_mape(y_true, y_pred),
    }

def http_get(url: str, params: Optional[dict] = None, timeout: int = 45) -> requests.Response:
    last = None
    for attempt in range(3):
        try:
            r = requests.get(
                url,
                params=params,
                timeout=timeout,
                headers={"User-Agent": "SaudiTourismPlanner-ML/2.0"},
            )
            if r.status_code in {429, 502, 503, 504} and attempt < 2:
                import time
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            last = exc
            if attempt < 2:
                import time
                time.sleep(2 ** attempt)
    raise RuntimeError(f"GET failed: {url}\n{last}")

print("✓ Utilities ready")

## 1. KAPSARC acquisition and validation

The notebook accepts a cached/local copy when available. Otherwise it downloads from the official KAPSARC portal. The parser explicitly validates the fields and filters only the POS **transaction value** indicator used by the forecast/context pipeline.

In [ ]:
def read_kapsarc_file(path: Path) -> pd.DataFrame:
    # The downloaded project copy is semicolon-separated. Official exports can
    # change delimiter, so try semicolon first and fall back to delimiter sniffing.
    try:
        df = pd.read_csv(path, sep=";")
        if df.shape[1] < 5:
            raise ValueError("single-column parse")
    except Exception:
        df = pd.read_csv(path, sep=None, engine="python")
    return df

def acquire_kapsarc() -> tuple[pd.DataFrame, dict]:
    if KAPSARC_LOCAL_PATH.exists():
        df = read_kapsarc_file(KAPSARC_LOCAL_PATH)
        return df, {
            "source": "cached official KAPSARC export",
            "path": str(KAPSARC_LOCAL_PATH),
            "retrieved_at_utc": utc_now_iso(),
        }

    errors = []
    for url in KAPSARC_EXPORT_CANDIDATES:
        try:
            response = http_get(url)
            raw_path = RAW_DIR / "kapsarc_download.csv"
            raw_path.write_bytes(response.content)
            df = read_kapsarc_file(raw_path)
            if df.shape[0] < 1000 or df.shape[1] < 5:
                raise ValueError(f"unexpected KAPSARC shape {df.shape}")
            shutil.copy2(raw_path, KAPSARC_LOCAL_PATH)
            return df, {
                "source": "KAPSARC official portal",
                "dataset_page": KAPSARC_DATASET_PAGE,
                "download_url": url,
                "retrieved_at_utc": utc_now_iso(),
            }
        except Exception as exc:
            errors.append({"url": url, "error": str(exc)})

    raise RuntimeError(
        "KAPSARC download failed. Place the official CSV at "
        f"{KAPSARC_LOCAL_PATH} and rerun.\n{json.dumps(errors, indent=2)}"
    )

kapsarc_raw, kapsarc_source_meta = acquire_kapsarc()

required_cols = {
    "Starting date", "Indicator", "Sectors", "City", "value (Multiple units)"
}
missing = required_cols - set(kapsarc_raw.columns)
if missing:
    raise ValueError(f"KAPSARC schema mismatch. Missing: {sorted(missing)}")

kapsarc_raw.to_csv(RAW_DIR / "kapsarc_raw_snapshot.csv", index=False)
save_json(kapsarc_source_meta, RAW_DIR / "kapsarc_source_metadata.json")

print("KAPSARC raw shape:", kapsarc_raw.shape)
display(kapsarc_raw.head())

In [ ]:
# ---------------------------------------------------------------------
# Correct KAPSARC granularity
# ---------------------------------------------------------------------
kapsarc = kapsarc_raw.copy()
kapsarc["Starting date"] = pd.to_datetime(kapsarc["Starting date"], errors="coerce")
kapsarc["value"] = pd.to_numeric(
    kapsarc["value (Multiple units)"], errors="coerce"
)

kapsarc = kapsarc[
    (kapsarc["Indicator"] == VALUE_INDICATOR)
    & kapsarc["Starting date"].notna()
    & kapsarc["value"].notna()
    & (kapsarc["value"] >= 0)
].copy()

kapsarc["month"] = kapsarc["Starting date"].dt.to_period("M").dt.to_timestamp()

# City signal: City != Total, Sectors == Total.
city_weekly = kapsarc[
    (kapsarc["Sectors"] == "Total")
    & (kapsarc["City"] != "Total")
    & (kapsarc["City"] != "OTHER")
].copy()
city_weekly["City"] = city_weekly["City"].map(CITY_NAME_MAP)
city_weekly = city_weekly.dropna(subset=["City"])

city_monthly = (
    city_weekly.groupby(["City", "month"], as_index=False)
    .agg(
        transaction_value_thousand_sar=("value", "sum"),
        weekly_obs=("value", "count"),
    )
)
city_monthly = city_monthly[
    city_monthly["weekly_obs"] >= MIN_WEEKLY_OBS_PER_MONTH
].copy()

# National tourism-sector signal: City == Total, selected tourism sectors.
sector_weekly = kapsarc[
    (kapsarc["City"] == "Total")
    & kapsarc["Sectors"].isin(TOURISM_SECTORS)
].copy()

sector_monthly = (
    sector_weekly.groupby(["Sectors", "month"], as_index=False)
    .agg(
        transaction_value_thousand_sar=("value", "sum"),
        weekly_obs=("value", "count"),
    )
)
sector_monthly = sector_monthly[
    sector_monthly["weekly_obs"] >= MIN_WEEKLY_OBS_PER_MONTH
].copy()
sector_monthly["place_group"] = sector_monthly["Sectors"].map(TOURISM_SECTORS)

print("City monthly:", city_monthly.shape)
print("Cities:", sorted(city_monthly["City"].unique()))
print("Complete range:", city_monthly["month"].min(), "→", city_monthly["month"].max())
print("Sector monthly:", sector_monthly.shape)
print("Tourism sectors:", sorted(sector_monthly["Sectors"].unique()))

assert city_monthly["month"].max() == sector_monthly["month"].max()
assert city_monthly["weekly_obs"].min() >= MIN_WEEKLY_OBS_PER_MONTH
assert sector_monthly["weekly_obs"].min() >= MIN_WEEKLY_OBS_PER_MONTH

### Why the incomplete-month filter matters

The raw source includes July 2025 with only one weekly observation in the supplied snapshot. Summing that partial month and treating it as comparable to full months would distort both model training and context normalization. A month is therefore retained only when it has at least four weekly observations.

In [ ]:
# EDA: verify incomplete latest raw month is excluded
raw_value = kapsarc[
    (kapsarc["Sectors"] == "Total")
    & (kapsarc["City"] != "Total")
    & (kapsarc["City"] != "OTHER")
].copy()

raw_month_counts = (
    raw_value.groupby("month")["Starting date"].nunique()
    .rename("weekly_dates")
    .reset_index()
)

display(raw_month_counts.tail(8))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(raw_month_counts["month"], raw_month_counts["weekly_dates"], marker="o")
ax.axhline(MIN_WEEKLY_OBS_PER_MONTH, linestyle="--", label="minimum complete-month observations")
ax.set_title("KAPSARC weekly observations per month")
ax.set_xlabel("Month")
ax.set_ylabel("Distinct weekly observations")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "kapsarc_month_completeness.png", dpi=160)
plt.show()

In [ ]:
# ---------------------------------------------------------------------
# Leakage-safe feature engineering
# ---------------------------------------------------------------------
def add_time_features(
    frame: pd.DataFrame,
    group_col: str,
    value_col: str = "transaction_value_thousand_sar",
) -> pd.DataFrame:
    out = frame.copy().sort_values([group_col, "month"]).reset_index(drop=True)
    out["group_key"] = out[group_col].map(normalize_text)
    grouped = out.groupby("group_key")[value_col]

    for lag in [1, 2, 3, 6, 12]:
        out[f"lag_{lag}"] = grouped.shift(lag)

    for window in [3, 6, 12]:
        out[f"rolling_{window}"] = grouped.transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()
        )
        out[f"rolling_std_{window}"] = grouped.transform(
            lambda s, w=window: s.shift(1).rolling(w, min_periods=2).std()
        )

    out["month_num"] = out["month"].dt.month
    out["month_sin"] = np.sin(2 * np.pi * out["month_num"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month_num"] / 12)
    out["year"] = out["month"].dt.year

    first_year = int(out["month"].dt.year.min())
    out["trend_months"] = (
        (out["month"].dt.year - first_year) * 12
        + out["month"].dt.month
    )
    return out

city_features = add_time_features(city_monthly, "City")
sector_features = add_time_features(sector_monthly, "Sectors")

# Final training rows need a real 12-month history so the seasonal-naive
# baseline is available for every evaluated row.
model_df = city_features.dropna(subset=["lag_12"]).copy()

city_features.to_csv(
    PROCESSED_DIR / "kapsarc_city_monthly_features.csv", index=False
)
sector_features.to_csv(
    PROCESSED_DIR / "kapsarc_sector_monthly_features.csv", index=False
)

print("Forecast-ready rows:", model_df.shape)
print("Forecast-ready range:", model_df["month"].min(), "→", model_df["month"].max())

## 2. Model selection with rolling-origin validation

The forecasting target is the **next monthly city-total POS transaction value**.

Candidate regressors are compared using three chronological rolling-origin validation folds. The primary selection metric is **MAE**. The seasonal-naive baseline (`lag_12`) is evaluated in every fold.

The final 12 complete months are left untouched until model selection is finished.

In [ ]:
# ---------------------------------------------------------------------
# Preprocessing + candidate models
# ---------------------------------------------------------------------
categorical_features = ["group_key"]
numeric_features = [f for f in FORECAST_FEATURES if f not in categorical_features]

def make_preprocessor():
    return ColumnTransformer(
        [
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                categorical_features,
            ),
            (
                "num",
                SimpleImputer(strategy="median"),
                numeric_features,
            ),
        ],
        remainder="drop",
    )

MODEL_CANDIDATES = {
    "ExtraTrees(n=300,leaf=1,max_features=1.0)": ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=1,
        max_features=1.0,
        n_jobs=-1,
        random_state=SEED,
    ),
    "ExtraTrees(n=300,leaf=2,max_features=1.0)": ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        max_features=1.0,
        n_jobs=-1,
        random_state=SEED,
    ),
    "ExtraTrees(n=300,leaf=3,max_features=sqrt)": ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        max_features="sqrt",
        n_jobs=-1,
        random_state=SEED,
    ),
    "RandomForest(n=300,leaf=2,max_features=sqrt)": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=SEED,
    ),
}

unique_months = np.array(sorted(model_df["month"].unique()))
if len(unique_months) <= FINAL_HOLDOUT_MONTHS + VALIDATION_FOLDS * VALIDATION_MONTHS_PER_FOLD:
    raise RuntimeError("Not enough complete historical months for the configured validation design.")

holdout_months = unique_months[-FINAL_HOLDOUT_MONTHS:]
holdout_start = pd.Timestamp(holdout_months[0])
pre_holdout = model_df[model_df["month"] < holdout_start].copy()

pre_months = np.array(sorted(pre_holdout["month"].unique()))
validation_block = VALIDATION_FOLDS * VALIDATION_MONTHS_PER_FOLD
validation_months = pre_months[-validation_block:]

folds = []
for fold_index in range(VALIDATION_FOLDS):
    start = fold_index * VALIDATION_MONTHS_PER_FOLD
    fold_months = validation_months[start:start + VALIDATION_MONTHS_PER_FOLD]
    val_start = pd.Timestamp(fold_months[0])
    val_end = pd.Timestamp(fold_months[-1])
    train_fold = pre_holdout[pre_holdout["month"] < val_start].copy()
    val_fold = pre_holdout[
        (pre_holdout["month"] >= val_start)
        & (pre_holdout["month"] <= val_end)
    ].copy()
    if train_fold.empty or val_fold.empty:
        raise RuntimeError("A rolling-origin fold is empty.")
    folds.append((val_start, val_end, train_fold, val_fold))

[(str(a.date()), str(b.date()), len(tr), len(va)) for a, b, tr, va in folds]

In [ ]:
# ---------------------------------------------------------------------
# Rolling-origin evaluation
# ---------------------------------------------------------------------
validation_rows = []

for model_name, regressor in MODEL_CANDIDATES.items():
    for fold_number, (val_start, val_end, train_fold, val_fold) in enumerate(folds, start=1):
        pipeline = Pipeline(
            [
                ("preprocessor", make_preprocessor()),
                ("regressor", clone(regressor)),
            ]
        )
        pipeline.fit(
            train_fold[FORECAST_FEATURES],
            train_fold["transaction_value_thousand_sar"],
        )
        prediction = pipeline.predict(val_fold[FORECAST_FEATURES])
        baseline = val_fold["lag_12"].to_numpy()

        model_m = regression_metrics(
            val_fold["transaction_value_thousand_sar"].to_numpy(),
            prediction,
        )
        baseline_m = regression_metrics(
            val_fold["transaction_value_thousand_sar"].to_numpy(),
            baseline,
        )

        validation_rows.append(
            {
                "model": model_name,
                "fold": fold_number,
                "validation_start": val_start,
                "validation_end": val_end,
                **{f"model_{k}": v for k, v in model_m.items()},
                **{f"baseline_{k}": v for k, v in baseline_m.items()},
            }
        )

validation_detail = pd.DataFrame(validation_rows)
validation_summary = (
    validation_detail.groupby("model", as_index=False)
    .agg(
        mean_mae=("model_mae", "mean"),
        mean_rmse=("model_rmse", "mean"),
        mean_r2=("model_r2", "mean"),
        mean_mape=("model_mape", "mean"),
        seasonal_naive_mean_mae=("baseline_mae", "mean"),
        seasonal_naive_mean_mape=("baseline_mape", "mean"),
    )
)
validation_summary["mae_improvement_vs_baseline"] = (
    validation_summary["seasonal_naive_mean_mae"]
    - validation_summary["mean_mae"]
) / validation_summary["seasonal_naive_mean_mae"]

validation_summary = validation_summary.sort_values(
    ["mean_mae", "mean_mape"], ascending=[True, True]
).reset_index(drop=True)

selected_model_name = str(validation_summary.iloc[0]["model"])
selected_regressor = MODEL_CANDIDATES[selected_model_name]

validation_detail.to_csv(
    METRICS_DIR / "rolling_origin_validation_detail.csv", index=False
)
validation_summary.to_csv(
    METRICS_DIR / "rolling_origin_validation_summary.csv", index=False
)

print("Selected model:", selected_model_name)
display(validation_summary)

selected_validation = validation_summary.iloc[0]
if selected_validation["mean_mae"] >= selected_validation["seasonal_naive_mean_mae"]:
    raise RuntimeError(
        "Production gate failed: the selected ML candidate did not beat the "
        "seasonal-naive baseline on rolling-origin validation."
    )

## 3. Untouched final chronological holdout

Only after model selection is complete do we evaluate the selected model on the last 12 complete months. This is the primary final historical test.

In [ ]:
train_final = model_df[model_df["month"] < holdout_start].copy()
test_final = model_df[model_df["month"].isin(holdout_months)].copy()

selected_pipeline = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        ("regressor", clone(selected_regressor)),
    ]
)
selected_pipeline.fit(
    train_final[FORECAST_FEATURES],
    train_final["transaction_value_thousand_sar"],
)

final_prediction = selected_pipeline.predict(test_final[FORECAST_FEATURES])
final_baseline = test_final["lag_12"].to_numpy()

final_model_metrics = regression_metrics(
    test_final["transaction_value_thousand_sar"].to_numpy(),
    final_prediction,
)
final_baseline_metrics = regression_metrics(
    test_final["transaction_value_thousand_sar"].to_numpy(),
    final_baseline,
)

final_improvement = (
    final_baseline_metrics["mae"] - final_model_metrics["mae"]
) / final_baseline_metrics["mae"]

print("Train:", train_final["month"].min(), "→", train_final["month"].max(), len(train_final))
print("Test :", test_final["month"].min(), "→", test_final["month"].max(), len(test_final))
print("Model:", final_model_metrics)
print("Seasonal naive:", final_baseline_metrics)
print("MAE improvement:", f"{final_improvement:.2%}")

assert train_final["month"].max() < test_final["month"].min()

test_predictions = test_final[
    ["month", "City", "transaction_value_thousand_sar", "lag_12"]
].copy()
test_predictions["model_prediction"] = final_prediction
test_predictions["seasonal_naive_prediction"] = final_baseline
test_predictions.to_csv(
    METRICS_DIR / "city_demand_final_holdout_predictions.csv", index=False
)

In [ ]:
# Final holdout plots
monthly_eval = (
    test_predictions.groupby("month", as_index=False)
    .agg(
        actual=("transaction_value_thousand_sar", "sum"),
        model=("model_prediction", "sum"),
        seasonal_naive=("seasonal_naive_prediction", "sum"),
    )
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_eval["month"], monthly_eval["actual"], marker="o", label="Actual")
ax.plot(monthly_eval["month"], monthly_eval["model"], marker="o", label="Selected ML model")
ax.plot(
    monthly_eval["month"],
    monthly_eval["seasonal_naive"],
    marker="o",
    linestyle="--",
    label="Seasonal-naive baseline",
)
ax.set_title("City-total demand forecast — untouched chronological holdout")
ax.set_xlabel("Month")
ax.set_ylabel("POS transaction value (thousand SAR)")
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "city_demand_final_holdout.png", dpi=160)
plt.show()

city_mae = (
    test_predictions.groupby("City")
    .apply(
        lambda g: pd.Series(
            {
                "model_mae": mean_absolute_error(
                    g["transaction_value_thousand_sar"], g["model_prediction"]
                ),
                "baseline_mae": mean_absolute_error(
                    g["transaction_value_thousand_sar"], g["seasonal_naive_prediction"]
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
city_mae["improvement"] = (
    city_mae["baseline_mae"] - city_mae["model_mae"]
) / city_mae["baseline_mae"]
city_mae.to_csv(METRICS_DIR / "city_level_holdout_mae.csv", index=False)
display(city_mae.sort_values("improvement", ascending=False))

## 4. Refit deployable model on all complete history

The final holdout metrics remain untouched and are saved as evaluation evidence. After evaluation, the deployable model is refit on all available forecast-ready rows so the production one-month forecast can use the most recent complete history.

In [ ]:
deployment_model = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        ("regressor", clone(selected_regressor)),
    ]
)
deployment_model.fit(
    model_df[FORECAST_FEATURES],
    model_df["transaction_value_thousand_sar"],
)

model_path = MODEL_DIR / "city_demand_model.joblib"
joblib.dump(deployment_model, model_path)

# Load immediately to prove serialization compatibility.
reloaded_model = joblib.load(model_path)
smoke_prediction = reloaded_model.predict(model_df[FORECAST_FEATURES].tail(10))
assert np.isfinite(smoke_prediction).all()

print("✓ Deployable model saved and reload-tested:", model_path)

## 5. Recommendation context artifacts

The recommendation engine does not misuse the forecast prediction as a venue rating.

Instead it uses:
- latest observed **city-total** POS demand score,
- latest observed **national tourism-sector** POS demand score,
- DataSaudi province/month occupancy seasonality.

The backend combines city and sector context as:

`regional_demand = 0.4 × city_score + 0.6 × sector_score`

In [ ]:
latest_complete_month = city_features["month"].max()

city_context = (
    city_features[city_features["month"] == latest_complete_month]
    [["City", "month", "transaction_value_thousand_sar"]]
    .rename(
        columns={
            "month": "latest_month",
            "transaction_value_thousand_sar": "latest_value",
        }
    )
    .copy()
)
city_context["city_demand_score"] = minmax01(city_context["latest_value"])
city_context["history_months"] = city_features.groupby("City")["month"].transform("count")[
    city_features["month"] == latest_complete_month
].to_numpy()
city_context["city_key"] = city_context["City"].map(normalize_text)

sector_context = (
    sector_features[sector_features["month"] == latest_complete_month]
    [["Sectors", "month", "transaction_value_thousand_sar", "place_group"]]
    .rename(
        columns={
            "month": "latest_month",
            "transaction_value_thousand_sar": "latest_value",
        }
    )
    .copy()
)
sector_context["sector_demand_score"] = minmax01(sector_context["latest_value"])
sector_context["history_months"] = sector_features.groupby("Sectors")["month"].transform("count")[
    sector_features["month"] == latest_complete_month
].to_numpy()

city_context.to_csv(PROCESSED_DIR / "city_demand_context.csv", index=False)
sector_context.to_csv(PROCESSED_DIR / "sector_demand_context.csv", index=False)

display(city_context.sort_values("city_demand_score", ascending=False))
display(sector_context.sort_values("sector_demand_score", ascending=False))

## 6. DataSaudi seasonality

The notebook first looks for a cached official DataSaudi 2024 occupancy workbook. If absent, it queries the public DataSaudi Tesseract API dynamically.

The resulting artifact is a normalized `province × month-of-year` occupancy seasonality table.

In [ ]:
# ---------------------------------------------------------------------
# DataSaudi local-workbook parser
# ---------------------------------------------------------------------
PROVINCE_CANONICAL = {
    "eastern": "Eastern Region",
    "albaha": "Al-Baha",
    "jouf": "Al-Jouf",
    "northern borders": "Northern Borders",
    "riyadh": "Al-Riyadh",
    "alqassim": "Al-Qaseem",
    "madinah": "Al-Madinah Al-Monawarah",
    "tabuk": "Tabouk",
    "jazan": "Jazan",
    "hail": "Hail",
    "aseer": "Aseer",
    "makkah": "Makkah Al-Mokarramah",
    "najran": "Najran",
}

def parse_datasaudi_xlsx(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="المناطق الادارية ", header=None)
    records = []
    current_region = None

    for row_index in range(5, len(raw)):
        english_region = raw.iloc[row_index, 17]
        if pd.notna(english_region):
            current_region = str(english_region).strip()

        if not current_region:
            continue

        accommodation_type = raw.iloc[row_index, 16]
        if pd.isna(accommodation_type):
            continue

        for month_num, column_index in enumerate(range(3, 15), start=1):
            value = pd.to_numeric(
                pd.Series([raw.iloc[row_index, column_index]]),
                errors="coerce",
            ).iloc[0]
            if pd.isna(value):
                continue
            records.append(
                {
                    "province": PROVINCE_CANONICAL.get(
                        normalize_text(current_region), current_region
                    ),
                    "month_num": month_num,
                    "occupancy_rate": float(value),
                    "accommodation_type": str(accommodation_type),
                }
            )

    frame = pd.DataFrame(records)
    if frame.empty:
        raise ValueError("DataSaudi workbook parser returned zero rows.")
    return frame

def datasaudi_cube_text(cube: dict) -> str:
    ann = cube.get("annotations", {}) or {}
    return normalize_text(
        " ".join(
            str(x)
            for x in [
                cube.get("name", ""),
                cube.get("caption", ""),
                ann.get("topic_en", ""),
                ann.get("subtopic_en", ""),
                ann.get("table_en", ""),
            ]
        )
    )

def cube_levels(cube: dict) -> list[str]:
    levels = []
    for dim in cube.get("dimensions", []) or []:
        for hierarchy in dim.get("hierarchies", []) or []:
            for level in hierarchy.get("levels", []) or []:
                if level.get("name"):
                    levels.append(level["name"])
    return levels

def cube_measures(cube: dict) -> list[str]:
    return [
        m["name"] for m in cube.get("measures", []) or []
        if m.get("name")
    ]

def choose_field(names: list[str], keywords: list[str], required=True):
    scored = []
    for name in names:
        key = normalize_text(name)
        score = sum(normalize_text(k) in key for k in keywords)
        if score:
            scored.append((score, name))
    if not scored:
        if required:
            raise ValueError(f"No field matching {keywords}; available={names}")
        return None
    return sorted(scored, reverse=True)[0][1]

def query_datasaudi_occupancy() -> tuple[pd.DataFrame, dict]:
    cubes_response = http_get(
        f"{DATASAUDI_API_BASE}/cubes",
        params={"locale": "en"},
    ).json()
    cubes = cubes_response.get("cubes", [])
    if not cubes:
        raise RuntimeError("DataSaudi returned no cubes.")

    scored = []
    for cube in cubes:
        text = datasaudi_cube_text(cube)
        score = (
            5 * ("occupancy" in text)
            + 4 * ("tourism" in text)
            + 3 * ("month" in text or "monthly" in text)
            + 2 * ("accommodation" in text)
        )
        if score:
            scored.append((score, cube))
    if not scored:
        raise RuntimeError("Could not identify a monthly tourism occupancy cube.")

    occupancy_cube = sorted(scored, key=lambda x: x[0], reverse=True)[0][1]
    levels = cube_levels(occupancy_cube)
    measures = cube_measures(occupancy_cube)

    province_level = choose_field(levels, ["province"])
    month_level = choose_field(levels, ["month"])
    accommodation_level = choose_field(levels, ["accommodation"], required=False)
    occupancy_measure = choose_field(measures, ["occupancy"])

    drilldowns = [province_level, month_level]
    if accommodation_level:
        drilldowns.append(accommodation_level)

    rows = []
    offset = 0
    page_size = 500
    while True:
        params = {
            "cube": occupancy_cube["name"],
            "locale": "en",
            "drilldowns": ",".join(drilldowns),
            "measures": occupancy_measure,
            "limit": f"{page_size},{offset}",
        }
        batch = http_get(
            f"{DATASAUDI_API_BASE}/data.jsonrecords",
            params=params,
        ).json().get("data", [])
        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size

    raw = pd.DataFrame(rows)
    if raw.empty:
        raise RuntimeError("DataSaudi occupancy query returned zero rows.")

    def find_col(preferred, keyword):
        if preferred in raw.columns:
            return preferred
        candidates = [c for c in raw.columns if keyword in normalize_text(c)]
        if not candidates:
            raise ValueError(f"Could not resolve {keyword} column.")
        return candidates[0]

    province_col = find_col(province_level, "province")
    month_col = find_col(month_level, "month")
    occupancy_col = find_col(occupancy_measure, "occupancy")

    frame = pd.DataFrame(
        {
            "province": raw[province_col].astype(str).str.strip(),
            "month_raw": raw[month_col],
            "occupancy_rate": pd.to_numeric(raw[occupancy_col], errors="coerce"),
        }
    )
    if frame["occupancy_rate"].dropna().median() > 1.5:
        frame["occupancy_rate"] = frame["occupancy_rate"] / 100.0

    parsed_month = pd.to_datetime(frame["month_raw"], errors="coerce")
    frame["month_num"] = parsed_month.dt.month

    if frame["month_num"].isna().mean() > 0.5:
        # Fallback for captions/IDs that contain a month number.
        extracted = frame["month_raw"].astype(str).str.extract(r"(?:^|[-/ ])(\d{1,2})(?:$|[-/ ])")[0]
        frame["month_num"] = pd.to_numeric(extracted, errors="coerce")

    frame = frame.dropna(subset=["province", "month_num", "occupancy_rate"])
    frame["month_num"] = frame["month_num"].astype(int)

    return frame, {
        "source": "DataSaudi Tesseract API",
        "cube": occupancy_cube["name"],
        "caption": occupancy_cube.get("caption"),
        "retrieved_at_utc": utc_now_iso(),
    }

if DATASAUDI_LOCAL_XLSX.exists():
    datasaudi_rows = parse_datasaudi_xlsx(DATASAUDI_LOCAL_XLSX)
    datasaudi_source_meta = {
        "source": "cached official DataSaudi occupancy workbook",
        "path": str(DATASAUDI_LOCAL_XLSX),
        "retrieved_at_utc": utc_now_iso(),
    }
else:
    datasaudi_rows, datasaudi_source_meta = query_datasaudi_occupancy()

datasaudi_rows["province_key"] = datasaudi_rows["province"].map(normalize_text)
seasonality_index = (
    datasaudi_rows.groupby(
        ["province", "province_key", "month_num"],
        as_index=False,
    )
    .agg(mean_occupancy=("occupancy_rate", "mean"))
)

seasonality_index["seasonality_score"] = (
    seasonality_index.groupby("province_key")["mean_occupancy"]
    .transform(minmax01)
)

seasonality_index.to_csv(
    PROCESSED_DIR / "datasaudi_seasonality_index.csv", index=False
)
save_json(
    datasaudi_source_meta,
    RAW_DIR / "datasaudi_source_metadata.json",
)

print("DataSaudi seasonality:", seasonality_index.shape)
display(seasonality_index.head(20))

In [ ]:
# DataSaudi EDA
national_seasonality = (
    seasonality_index.groupby("month_num", as_index=False)
    .agg(mean_occupancy=("mean_occupancy", "mean"))
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    national_seasonality["month_num"],
    national_seasonality["mean_occupancy"],
    marker="o",
)
ax.set_title("DataSaudi tourism occupancy seasonality")
ax.set_xlabel("Month")
ax.set_ylabel("Mean occupancy rate")
ax.set_xticks(range(1, 13))
fig.tight_layout()
fig.savefig(PLOTS_DIR / "datasaudi_seasonality.png", dpi=160)
plt.show()

## 7. Metrics, metadata, and backend-ready artifacts

In [ ]:
metrics = {
    "problem": "monthly city-total POS demand forecasting",
    "incomplete_month_policy": (
        f"months with fewer than {MIN_WEEKLY_OBS_PER_MONTH} weekly observations are excluded"
    ),
    "kapsarc_granularity_note": (
        "KAPSARC provides city totals and national sector totals as separate cuts; "
        "the pipeline does not claim observed city×sector transactions."
    ),
    "selection_method": (
        f"{VALIDATION_FOLDS} rolling-origin validation folds; primary metric MAE"
    ),
    "selected_model": selected_model_name,
    "validation": {
        "selected_model_mean_mae": float(selected_validation["mean_mae"]),
        "selected_model_mean_mape": float(selected_validation["mean_mape"]),
        "seasonal_naive_mean_mae": float(
            selected_validation["seasonal_naive_mean_mae"]
        ),
        "mae_improvement_vs_baseline": float(
            selected_validation["mae_improvement_vs_baseline"]
        ),
    },
    "final_holdout": {
        "train_start": str(train_final["month"].min().date()),
        "train_end": str(train_final["month"].max().date()),
        "test_start": str(test_final["month"].min().date()),
        "test_end": str(test_final["month"].max().date()),
        "rows_train": int(len(train_final)),
        "rows_test": int(len(test_final)),
        "model": final_model_metrics,
        "seasonal_naive_baseline": final_baseline_metrics,
        "mae_improvement_vs_baseline": float(final_improvement),
    },
}
save_json(metrics, METRICS_DIR / "city_demand_metrics.json")

metadata = {
    "model_version": RUN_ID,
    "built_at_utc": utc_now_iso(),
    "kapsarc_source": "Point of Sale Transactions by Sector and City",
    "kapsarc_granularity": {
        "city_signal": "City != Total with Sectors == Total",
        "sector_signal": "City == Total with tourism-relevant sector",
        "observed_city_sector_cross": "not available in source",
    },
    "forecast_model": {
        "artifact": "city_demand_model.joblib",
        "selected_model": selected_model_name,
        "features": FORECAST_FEATURES,
        "forecast_horizon": (
            "one month ahead from latest complete observed city history"
        ),
        "deployment_refit": "all available model rows after holdout evaluation",
        "metrics_file": "city_demand_metrics.json",
    },
    "recommendation_context": {
        "city_context": "city_demand_context.csv",
        "sector_context": "sector_demand_context.csv",
        "seasonality_context": "datasaudi_seasonality_index.csv",
        "regional_demand_formula": (
            "0.4 * city_demand_score + 0.6 * sector_demand_score"
        ),
        "rank_weights": RANK_WEIGHTS,
        "budget_in_rank": False,
        "accessibility_in_rank_without_confirmed_evidence": False,
    },
    "cost_estimates_sar": {
        "attraction": 50.0,
        "restaurant": 80.0,
        "cafe": 35.0,
    },
    "limitations": [
        (
            "KAPSARC does not expose city-by-sector transactions in the source "
            "table; city and sector signals are combined as contextual features, "
            "not represented as observed city-sector spend."
        ),
        (
            "Venue-level price is not provided by the current normalized Places "
            "API; itinerary costs are category-level estimates and must be labeled."
        ),
        (
            "Confirmed accessibility is usually absent from the current normalized "
            "Places API; strict accessibility only accepts confirmed evidence."
        ),
        (
            "Opening hours are sparse and unknown values are never treated as "
            "confirmed open."
        ),
        (
            "Routing uses Haversine distance with a documented road-distance "
            "factor unless a routing service is added later."
        ),
        (
            "Human-labeled recommendation relevance and user satisfaction require "
            "a real user study and are not fabricated."
        ),
    ],
}
save_json(metadata, METRICS_DIR / "deployment_metadata.json")

print(json.dumps(metrics, indent=2))

In [ ]:
# Copy deployable inference artifacts
artifact_sources = {
    MODEL_DIR / "city_demand_model.joblib": "city_demand_model.joblib",
    PROCESSED_DIR / "kapsarc_city_monthly_features.csv": "kapsarc_city_monthly_features.csv",
    PROCESSED_DIR / "kapsarc_sector_monthly_features.csv": "kapsarc_sector_monthly_features.csv",
    PROCESSED_DIR / "city_demand_context.csv": "city_demand_context.csv",
    PROCESSED_DIR / "sector_demand_context.csv": "sector_demand_context.csv",
    PROCESSED_DIR / "datasaudi_seasonality_index.csv": "datasaudi_seasonality_index.csv",
    METRICS_DIR / "city_demand_metrics.json": "city_demand_metrics.json",
    METRICS_DIR / "deployment_metadata.json": "deployment_metadata.json",
}

for src, filename in artifact_sources.items():
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, BACKEND_ARTIFACTS_DIR / filename)

# Refresh latest_backend_artifacts atomically enough for a notebook workflow.
for old in LATEST_BACKEND_ARTIFACTS_DIR.iterdir():
    if old.is_file():
        old.unlink()
for src in BACKEND_ARTIFACTS_DIR.iterdir():
    if src.is_file():
        shutil.copy2(src, LATEST_BACKEND_ARTIFACTS_DIR / src.name)

bundle_path = RUN_DIR / "backend_ml_artifacts.zip"
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in BACKEND_ARTIFACTS_DIR.iterdir():
        if path.is_file():
            zf.write(path, arcname=path.name)

print("✓ Backend artifact bundle:", bundle_path)

## 8. Production smoke tests

These tests use the artifacts produced in this run. They verify serialization, finite forecasts, correct chronology, expected granularity, and DataSaudi/context availability.

In [ ]:
# Model reload and one-month forecast smoke tests
deployed = joblib.load(BACKEND_ARTIFACTS_DIR / "city_demand_model.joblib")
deployed_city_features = pd.read_csv(
    BACKEND_ARTIFACTS_DIR / "kapsarc_city_monthly_features.csv"
)
deployed_city_features["month"] = pd.to_datetime(
    deployed_city_features["month"], errors="coerce"
)

smoke_rows = []
for city in sorted(deployed_city_features["City"].dropna().unique()):
    subset = deployed_city_features[
        deployed_city_features["City"] == city
    ].sort_values("month")
    history = pd.to_numeric(
        subset["transaction_value_thousand_sar"], errors="coerce"
    ).dropna()
    if len(history) < 12:
        continue

    target_month = subset["month"].max() + pd.offsets.MonthBegin(1)
    first_year = int(deployed_city_features["month"].dt.year.min())
    row = pd.DataFrame(
        [
            {
                "group_key": normalize_text(city),
                "year": int(target_month.year),
                "month_num": int(target_month.month),
                "month_sin": float(
                    np.sin(2 * np.pi * target_month.month / 12)
                ),
                "month_cos": float(
                    np.cos(2 * np.pi * target_month.month / 12)
                ),
                "trend_months": int(
                    (target_month.year - first_year) * 12
                    + target_month.month
                ),
                "lag_1": float(history.iloc[-1]),
                "lag_2": float(history.iloc[-2]),
                "lag_3": float(history.iloc[-3]),
                "lag_6": float(history.iloc[-6]),
                "lag_12": float(history.iloc[-12]),
                "rolling_3": float(history.iloc[-3:].mean()),
                "rolling_6": float(history.iloc[-6:].mean()),
                "rolling_12": float(history.iloc[-12:].mean()),
                "rolling_std_3": float(history.iloc[-3:].std(ddof=1)),
                "rolling_std_6": float(history.iloc[-6:].std(ddof=1)),
                "rolling_std_12": float(history.iloc[-12:].std(ddof=1)),
            }
        ]
    )
    pred = float(deployed.predict(row[FORECAST_FEATURES])[0])
    assert np.isfinite(pred)
    smoke_rows.append(
        {
            "city": city,
            "forecast_month": target_month.strftime("%Y-%m"),
            "prediction_thousand_sar": max(0.0, pred),
            "seasonal_naive": float(history.iloc[-12]),
        }
    )

smoke_forecasts = pd.DataFrame(smoke_rows)
smoke_forecasts.to_csv(
    METRICS_DIR / "deployment_smoke_forecasts.csv", index=False
)
display(smoke_forecasts)

# Hard gates
assert not city_context.empty
assert not sector_context.empty
assert not seasonality_index.empty
assert set(sector_context["place_group"]) == {
    "accommodation", "attractions", "food"
}
assert train_final["month"].max() < test_final["month"].min()
assert city_features["month"].max() < kapsarc["month"].max() or (
    city_features["month"].max() == kapsarc["month"].max()
)
assert metrics["final_holdout"]["model"]["r2"] <= 1.0
assert metrics["validation"]["mae_improvement_vs_baseline"] > 0

print("✓ Production smoke tests passed")

In [ ]:
# ---------------------------------------------------------------------
# Final report and manifest
# ---------------------------------------------------------------------
report_lines = [
    "# Saudi Tourism Planner — Production ML Run",
    "",
    f"Run ID: {RUN_ID}",
    f"Generated UTC: {utc_now_iso()}",
    "",
    "## KAPSARC",
    f"- Raw rows: {len(kapsarc_raw):,}",
    f"- Value-indicator rows: {len(kapsarc):,}",
    f"- Complete city-month rows: {len(city_monthly):,}",
    f"- Complete city history through: {city_monthly['month'].max().date()}",
    "- City × sector observed cross: NOT available in the source table",
    "",
    "## Forecast model",
    f"- Selected: {selected_model_name}",
    f"- Rolling validation MAE: {selected_validation['mean_mae']:.2f}",
    f"- Rolling validation improvement vs seasonal-naive: {selected_validation['mae_improvement_vs_baseline']:.2%}",
    f"- Holdout MAE: {final_model_metrics['mae']:.2f}",
    f"- Holdout RMSE: {final_model_metrics['rmse']:.2f}",
    f"- Holdout R²: {final_model_metrics['r2']:.4f}",
    f"- Holdout MAPE: {final_model_metrics['mape']:.2%}",
    f"- Holdout MAE improvement vs seasonal-naive: {final_improvement:.2%}",
    "",
    "## Recommendation context",
    f"- City demand context rows: {len(city_context)}",
    f"- National tourism-sector context rows: {len(sector_context)}",
    f"- DataSaudi seasonality rows: {len(seasonality_index)}",
    "",
    "## Scientific integrity",
    "- Venue price is not claimed as verified; itinerary cost is estimated.",
    "- Unknown accessibility is not treated as confirmed accessible.",
    "- Unknown opening hours are not treated as confirmed open.",
    "- Routing is an explicit estimate unless a real routing provider is integrated.",
    "- Human relevance / satisfaction metrics require real user labels and are not fabricated.",
]
report = "\n".join(report_lines)
(REPORTS_DIR / "ML_PRODUCTION_RUN_REPORT.md").write_text(
    report, encoding="utf-8"
)

manifest_files = sorted(
    str(p.relative_to(RUN_DIR))
    for p in RUN_DIR.rglob("*")
    if p.is_file()
)
manifest = {
    "run_id": RUN_ID,
    "generated_at_utc": utc_now_iso(),
    "runtime": RUNTIME,
    "artifact_count": len(manifest_files),
    "files": manifest_files,
}
save_json(manifest, RUN_DIR / "manifest.json")

print(report)
print("\n✓ COMPLETE")
print("Run directory:", RUN_DIR)
print("Latest backend artifacts:", LATEST_BACKEND_ARTIFACTS_DIR)
print("Artifact ZIP:", bundle_path)